In [ ]:
# (C) Martin Reißel

from sympy import *

from IPython.display import display, Math, Latex
from sympy.interactive import printing
printing.init_printing(use_latex='mathjax')
platex = lambda A: latex(A,mat_str='pmatrix',mat_delim='')

import numpy as np

%matplotlib inline

# Jacobi-Verfahren zur Eigenwertberechnung

Berechnen Sie mit dem Jacobi-Verfahren Nähreungen der Eigenwerte der Matrix

In [ ]:
A = Matrix([[3,4],[4,3]])

Math('A=' + platex(A))

# Lösung

Durch wiederholte Ähnlichkeits-Transformationen mit
Drehmatrizen $Q_{i_0 j_0}$ soll $A$ auf näherungsweise Diagonalgestalt
transformiert werden, d.h.
$$
\tilde{A} 
=
Q_{i_0 j_0}^{-1} A Q_{i_0 j_0}
=
Q_{i_0 j_0}^T A Q_{i_0 j_0}
$$
wobei $a_{i_0 j_0}$, $j_0>i_0$, das betragsgrößte Nebendiagonalelement
von $A$ ist und $c,s$ gegeben sind durch
$$
\begin{aligned}
c &= \sqrt{\frac{1}{2} + \frac{1}{2}\sqrt{\frac{\alpha^2}{1+\alpha^2}}},
&
s &=\frac{\text{sign}(\alpha)}{2c\sqrt{1+\alpha^2}},
\\[2ex]
\alpha &=
\frac{a_{j_0j_0} - a_{i_0i_0}}{2a_{i_0j_0}},
&
\text{sign}(t)&=\begin{cases} 1 & \text{für }t\ge 0 \\ -1 &t<0\end{cases}
\end{aligned}
$$

Angewandt auf die gegebene Matrix folgt

In [ ]:
def Vor(x):
    if x<0:
        return -1
    else:
        return 1

def MaxNd(A):
    Anabs = abs(np.triu(A, 1))
    return np.unravel_index(np.argmax(Anabs), Anabs.shape)


def Jrot(A, itmax=1):
    Ak = A.copy()
    
    n = Ak.shape[0] 

    Qk = eye(n)
    
    nit = 0
    
    for nit in range(itmax):
        [i0,j0] = MaxNd(Ak)
        
        alpha   = (Ak[j0,j0] - Ak[i0,i0])/(2 * Ak[i0,j0])
        c       = sqrt( (1 + sqrt(alpha**2 / (1 + alpha**2) ) ) / 2)
        s       = Vor(alpha)/(2 * c * sqrt(1 + alpha**2))
        
        Q0 = eye(n)
        Q0[i0,i0] = c
        Q0[j0,j0] = c
        Q0[i0,j0] = s
        Q0[j0,i0] = -s
        
        Akalt = Ak
        Ak = Q0.T * Ak * Q0
        Qk = Qk * Q0
        
        #print nit, i0, j0, c, s, SumNd(Ak)
        
        display(Math(r'A_k=' + platex(Akalt) 
                     + r',\quad i_0,j_0 = {},{}'.format(i0+1,j0+1)
                     + r',\quad c,s,\alpha = {},{},{}'.format(latex(c), latex(s), latex(alpha))
                     + r',\quad Q_k = ' + platex(Q0)
                     + r',\quad A_{k+1} =  Q_k^T A_k Q_k =' + platex(Ak)
                    ))
    
    ew = Matrix(np.diag(Ak)).T
    
    return (ew, Qk, Ak)

ew, ev, _ = Jrot(A)

Damit erhalten wir als Näherungen für Eigenwerte und Eigenvektoren

In [ ]:
for k in range(len(ew)):
    display(Math('\lambda_{}={},\qquad v_{}={}'.format(k+1, latex(ew[k]), k+1, platex(ev[:,k]))))